## Reading images with the ground truth 

### Pre- processing the ImHANDWRTITNG dataset 1 : metadata creation

In [1]:
import os
import csv
import concurrent.futures
from collections import defaultdict

# Paths
base_folder = "/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/"  # This should contain formsA-D, formsE-H, ...
lines_txt_path = "/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/ascii/lines.txt"  # Adjust if needed
output_csv = "Handwriting_1_matadata.csv"

# Step 1: Load and parse lines.txt
form_lines = defaultdict(list)

with open(lines_txt_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.startswith("#") or not line.strip():
            continue
        parts = line.strip().split()
        line_id = parts[0]
        form_id = "-".join(line_id.split("-")[:2])
        text = " ".join(parts[8:]).replace("|", " ")
        form_lines[form_id].append(text)

# Step 2: Scan all folders for image paths
def scan_for_images(folder):
    img_paths = {}
    for root, _, files in os.walk(folder):
        for file in files:
            if file.endswith((".png", ".jpeg", ".jpg")):
                form_id = os.path.splitext(file)[0]
                img_paths[form_id] = os.path.join(root, file)
    return img_paths

# Use multithreading for scanning
form_image_paths = {}

with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [executor.submit(scan_for_images, os.path.join(base_folder, subfolder))
               for subfolder in os.listdir(base_folder)
               if subfolder.startswith("forms")]
    for future in concurrent.futures.as_completed(futures):
        form_image_paths.update(future.result())

# Step 3: Build dataset
dataset = []

for form_id, lines in form_lines.items():
    if form_id in form_image_paths:
        image_path = form_image_paths[form_id]
        text = " ".join(lines)
        dataset.append([form_id, image_path, text])

# Step 4: Write to CSV
with open(output_csv, "w", newline="", encoding="utf-8") as out_file:
    writer = csv.writer(out_file)
    writer.writerow(["form_id", "image_path", "transcription"])
    writer.writerows(dataset)

print(f"Dataset saved to: {output_csv}")

Dataset saved to: Handwriting_1_matadata.csv


## Preprocessing image

In [3]:
import cv2
import numpy as np
import os

def resize_by_width_auto_scale(
    image_path,
    target_width=512,
    max_height=128,
    binarize=False
):
    """
    Resize image to fixed width, auto-scale to fit max height.
    Pads if needed. Returns normalized [1 x H x W] image.
    """
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")

    h, w = img.shape
    scale_w = target_width / w
    new_h = int(h * scale_w)

    # Resize to fixed width
    img = cv2.resize(img, (target_width, new_h), interpolation=cv2.INTER_AREA)

    # If height exceeds max_height, downscale everything again
    if new_h > max_height:
        scale_h = max_height / new_h
        new_w = int(target_width * scale_h)
        img = cv2.resize(img, (new_w, max_height), interpolation=cv2.INTER_AREA)

        # Pad right side if new width is smaller
        padded = np.ones((max_height, target_width), dtype=np.uint8) * 255
        padded[:, :new_w] = img
        img = padded
    else:
        # Pad bottom if height is short
        padded = np.ones((max_height, target_width), dtype=np.uint8) * 255
        padded[:new_h, :] = img
        img = padded

    if binarize:
        img = cv2.adaptiveThreshold(
            img, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, 15, 10
        )

    # Normalize to [0, 1]
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)  # [1 x H x W]
    return img


def process_folder_by_width(
    input_folder,
    output_folder,
    target_width=512,
    max_height=128,
    binarize=False,
    out_ext=".png"
):
    """
    Resize + pad all images in a folder using width-based strategy.
    Saves resized images to output_folder.
    """
    os.makedirs(output_folder, exist_ok=True)

    for fname in os.listdir(input_folder):
        if fname.lower().endswith((".png", ".jpeg", ".jpg")):
            try:
                input_path = os.path.join(input_folder, fname)
                img = resize_by_width_auto_scale(
                    image_path=input_path,
                    target_width=target_width,
                    max_height=max_height,
                    binarize=binarize
                )

                output_name = os.path.splitext(fname)[0] + out_ext
                output_path = os.path.join(output_folder, output_name)
                save_img = (img[0] * 255).astype(np.uint8)
                cv2.imwrite(output_path, save_img)
                print(f"✅ Saved: {output_name}")
            except Exception as e:
                print(f"❌ Failed on {fname}: {e}")

In [9]:
process_folder_by_width(
    input_folder="/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/formsA-D",
    output_folder="/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/processed/formsA2D",
    target_width=256,
    max_height=128,
    binarize= False
)

✅ Saved: a04-006.png
✅ Saved: c03-087e.png
✅ Saved: a02-057.png
✅ Saved: d07-082.png
✅ Saved: d07-096.png
✅ Saved: a05-017.png
✅ Saved: b04-026.png
✅ Saved: d06-050.png
✅ Saved: c06-020.png
❌ Failed on d04-117 (1).png: Image not found: /Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/formsA-D/d04-117 (1).png
✅ Saved: d05-030.png
✅ Saved: c02-000.png
✅ Saved: d04-021.png
✅ Saved: b06-042.png
✅ Saved: b06-056.png
✅ Saved: d04-008.png
✅ Saved: c04-044.png
❌ Failed on d04-032 (1).png: Image not found: /Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/formsA-D/d04-032 (1).png
✅ Saved: c04-050.png
✅ Saved: d05-025.png
✅ Saved: b02-102.png
✅ Saved: a01-049x.png
✅ Saved: b04-147.png
✅ Saved: d06-086.png
✅ Saved: a03-047.png
❌ Failed on d03-112 (1).png: Image not found: /Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/formsA-D/d03-112 (1).png
✅ Saved: a02-042.png
❌ Failed on d04-050 (1).png: Image not found: /Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/formsA-D/d04-050 (1

In [6]:
process_folder_by_width(
    input_folder="/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/formsE-H",
    output_folder="/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/processed/formsE2H",
    target_width=256,
    max_height=128,
    binarize=False
)

✅ Saved: f02-033.png
✅ Saved: g06-037m.png
✅ Saved: g06-042p.png
✅ Saved: g06-042g.png
✅ Saved: e04-095.png
✅ Saved: h04-055.png
✅ Saved: e01-107.png
✅ Saved: e01-113.png
✅ Saved: h02-004.png
✅ Saved: h04-082.png
✅ Saved: f07-016.png
✅ Saved: e06-033.png
✅ Saved: f07-002.png
✅ Saved: f01-053.png
✅ Saved: g07-022a.png
✅ Saved: g06-105.png
✅ Saved: g03-040.png
✅ Saved: e06-026.png
✅ Saved: f01-085.png
✅ Saved: g04-014.png
✅ Saved: h07-020.png
✅ Saved: f07-046b.png
✅ Saved: g01-031.png
✅ Saved: g07-014b.png
✅ Saved: g01-025.png
✅ Saved: e04-043.png
✅ Saved: g01-019.png
✅ Saved: g07-018a.png
✅ Saved: g06-018a.png
✅ Saved: g06-042f.png
✅ Saved: g06-037l.png
✅ Saved: f02-030.png
✅ Saved: g06-037n.png
✅ Saved: f04-061.png
✅ Saved: f04-049.png
✅ Saved: g06-042d.png
✅ Saved: g06-018c.png
✅ Saved: g01-027.png
✅ Saved: g07-079a.png
✅ Saved: f03-169.png
✅ Saved: f03-182.png
✅ Saved: e04-109.png
✅ Saved: e06-030.png
✅ Saved: g07-022b.png
✅ Saved: h07-028a.png
✅ Saved: f04-100.png
✅ Saved: g03-043.p

In [7]:
process_folder_by_width(
    input_folder="/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/formsI-Z",
    output_folder="/Users/krishnanand/Documents/Git/Data/IMHANDWRITING_1/processed/formsI2Z",
    target_width=256,
    max_height=128,
    binarize=False
)

✅ Saved: k04-061.png
✅ Saved: k04-075.png
✅ Saved: l07-111.png
✅ Saved: k02-018.png
✅ Saved: l04-005.png
✅ Saved: p03-163.png
✅ Saved: r06-111.png
✅ Saved: m04-072.png
✅ Saved: n06-194.png
✅ Saved: k04-115.png
✅ Saved: l07-065.png
✅ Saved: n04-052.png
✅ Saved: l04-159.png
✅ Saved: r02-131.png
✅ Saved: l04-170.png
✅ Saved: k01-051.png
✅ Saved: n04-084.png
✅ Saved: m03-033.png
✅ Saved: n02-016.png
✅ Saved: r06-070.png
✅ Saved: m01-136.png
✅ Saved: n06-156.png
✅ Saved: m04-107.png
✅ Saved: m04-113.png
✅ Saved: r03-096.png
✅ Saved: p03-189.png
✅ Saved: m01-095.png
✅ Saved: r02-078.png
✅ Saved: l07-138.png
✅ Saved: l04-012.png
✅ Saved: k07-176.png
✅ Saved: l01-143.png
✅ Saved: l01-157.png
✅ Saved: r06-106.png
✅ Saved: j04-015.png
✅ Saved: n06-140.png
✅ Saved: r06-066.png
✅ Saved: n04-092.png
✅ Saved: n02-000.png
✅ Saved: n02-028.png
✅ Saved: l01-023.png
✅ Saved: r02-127.png
✅ Saved: n04-044.png
✅ Saved: k04-103.png
✅ Saved: p03-029.png
✅ Saved: k03-157.png
✅ Saved: n06-182.png
✅ Saved: m01-

## Modelling

## Language Detection

In [1]:
import fasttext
import torch

# Detect the best available device
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Ensure correct path to model file
model_path = "/Users/krishnanand/Documents/Git/Dependencies/lid.176.bin"  # Update this path if necessary
model = fasttext.load_model(model_path)

def detect_language(text):
    """Detect language of given text using FastText."""
    predictions = model.predict(text)
    lang_code = predictions[0][0].replace('__label__', '')
    return lang_code

text = "Bonjour, comment allez-vous?"
detected_lang = detect_language(text)
print(f"Detected Language: {detected_lang}")

## Summarization  

In [2]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else (0 if torch.backends.mps.is_available() else -1)

# Load summarization pipeline (T5-Base)
summarizer = pipeline("summarization", model="t5-base", device=device)

# Example input text
text = """
Artificial intelligence (AI) is the simulation of human intelligence processes by machines, especially computer systems. 
These processes include learning, reasoning, and self-correction. AI is one of the most exciting and disruptive technologies 
of our time, influencing everything from healthcare to transportation.
"""

# Generate summary
summary = summarizer(text, max_length=100, min_length=30, do_sample=False)
summary_text = summary[0]['summary_text']
print("Summary in English:", summary_text)

# Load translation pipeline with a better model
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-es", device=device)

# Translate the summary
translated_summary = translator(summary_text)
print("Translated Summary in Spanish:", translated_summary[0]['translation_text'])

## Translator

In [3]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else (0 if torch.backends.mps.is_available() else -1)

# Load the NLLB model for French to Hindi translation
translator = pipeline("translation", model="facebook/nllb-200-distilled-600M", device=device)

# Example French text
text = "Bonjour, comment allez-vous aujourd'hui?"

# Translate the text from French to English
translated_text = translator(text, src_lang="fra_Latn", tgt_lang="eng_Latn")
english_translation = translated_text[0]['translation_text']
print("Translated to English:", english_translation)

# Now, translate from English to Hindi
translated_to_hindi = translator(english_translation, src_lang="eng_Latn", tgt_lang="hin_Deva")
print("Translated to Hindi:", translated_to_hindi[0]['translation_text'])

## TTS

In [4]:
from TTS.api import TTS
import torch

# Detect best available device (CUDA for NVIDIA, MPS for Apple Silicon, CPU as fallback)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Load a single-language English TTS model and move it to the selected device
tts = TTS("tts_models/en/ljspeech/tacotron2-DDC").to(device)

# Text to convert
text = ("The error 'Model is not multi-lingual but language is provided' means that "
        "the model does not support multiple languages. This model can only generate speech in English, "
        "so specifying language (French) is invalid.")

# Convert text to speech and save as an audio file
tts.tts_to_file(text=text, file_path="/Users/krishnanand/Documents/Git/Tests/test123.wav")

print("Speech synthesis complete! Output saved as 'test123.wav'.")

In [1]:
pip install colabcode

In [3]:
%pip install colabcode